In [1]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

In [3]:
df = df.sample(n=50_000, random_state=42)

In [4]:
df = df[df['TP_ESCOLA'].isin([2,3])]
df = df[df['TP_ESTADO_CIVIL'].isin([1,2,3,4])]

In [5]:
df['TP_LOCALIZACAO_ESC'] = df['TP_LOCALIZACAO_ESC'].fillna(0)
df['TP_DEPENDENCIA_ADM_ESC'] = df['TP_DEPENDENCIA_ADM_ESC'].fillna(0)
df['TP_SIT_FUNC_ESC'] = df['TP_SIT_FUNC_ESC'].fillna(0)

In [6]:
questionario = [f'Q{str(i).zfill(3)}' for i in range(1, 26) if i != 5]
df = df.dropna(subset=questionario)

In [7]:
FALTOU = (
    (df['TP_PRESENCA_CH'] != 1) | 
    (df['TP_PRESENCA_LC'] != 1) | 
    (df['TP_PRESENCA_CN'] != 1) | 
    (df['TP_PRESENCA_MT'] != 1)
)

df['FALTOU'] = FALTOU.astype(int)

In [8]:
nominais = ['TP_ESCOLA', 'TP_DEPENDENCIA_ADM_ESC',
            'TP_ESTADO_CIVIL', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC']

ordinais = ['TP_FAIXA_ETARIA', 'TP_ST_CONCLUSAO']

categorias_quest = []
for i in range(1, 26):
    if i == 5:
        continue
    elif i == 6:
        categorias_quest.append(list('ABCDEFGHIJKLMNOPQ'))  
    elif i == 25:
        categorias_quest.append(list('AB'))                  
    elif i in [1, 2]:
        categorias_quest.append(list('ABCDEFGH'))            
    elif i in [3, 4]:
        categorias_quest.append(list('ABCDEF'))              
    else:
        categorias_quest.append(list('ABCDE'))               

binarias = ['IN_TREINEIRO']

In [9]:
pipe_nominal = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

pipe_ordinal = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

pipe_quest = OrdinalEncoder(
    categories=categorias_quest,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

pipe_continua = StandardScaler()

preprocessor = ColumnTransformer(transformers=[
    ('nominal',      pipe_nominal,  nominais),
    ('ordinal',      pipe_ordinal,  ordinais),
    ('questionario', pipe_quest,    questionario),
    ('binaria',      'passthrough', binarias),
], remainder='drop')

In [10]:
X = df[colunas]
y = df['FALTOU']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [12]:
X_train = preprocessor.fit_transform(X_train).astype(np.float32)
X_val   = preprocessor.transform(X_val).astype(np.float32)   
X_test  = preprocessor.transform(X_test).astype(np.float32) 

In [13]:
pipe = Pipeline([
    ('svm', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

params = {
    'svm__C': [0.1, 1, 10],
    'svm__gamma': ['scale', 'auto']
}

grid = GridSearchCV(pipe, params, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)  

print(grid.best_params_)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
{'svm__C': 1, 'svm__gamma': 'scale'}


In [14]:
print(classification_report(y_val, grid.predict(X_val)))

              precision    recall  f1-score   support

           0       0.85      0.56      0.68      1748
           1       0.38      0.72      0.49       636

    accuracy                           0.61      2384
   macro avg       0.61      0.64      0.59      2384
weighted avg       0.72      0.61      0.63      2384

